# Проверка покрытия ССП (zo) для ИНН из февральского отчета

Цель:
- взять все ИНН из Excel-отчета коллег за февраль;
- проверить, у скольких ИНН проставляется `zo` (ССП) в `sandbox_ai.kedr2hive__dbo__active_clients_ul_new`;
- выделить ИНН без ССП и ИНН с конфликтом по ССП (несколько `zo`).


In [ ]:
import re
import pandas as pd
from decimal import Decimal, InvalidOperation
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None


In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'

ssp_table = 'sandbox_ai.kedr2hive__dbo__active_clients_ul_new'
ssp_inn_col = 'inn'
ssp_zo_col = 'zo'

sample_size = 200

print('excel_path =', excel_path)
print('ssp_table =', ssp_table)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute(f'invalidate metadata {ssp_table}')

print('Impala initialized')


In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if s == '':
        return None
    return s

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def detect_excel_header_row(path, required_cols, scan_rows=40):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 1:
            return i
    return 0


## 1) ИНН из Excel (февраль)


In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Колонка ИНН не найдена в Excel: {excel_inn_col}')

excel_df = excel_raw[[excel_inn_col]].copy()
excel_df.columns = ['inn_raw']
excel_df['inn_norm'] = excel_df['inn_raw'].apply(normalize_inn)

excel_inn_all = excel_df[excel_df['inn_norm'].notna()].copy()
excel_inn_unique = excel_inn_all[['inn_norm']].drop_duplicates().copy()

stats_excel = pd.DataFrame([
    {'metric': 'excel_rows_total', 'value': len(excel_df)},
    {'metric': 'excel_rows_with_valid_inn', 'value': len(excel_inn_all)},
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_unique)},
    {'metric': 'excel_invalid_inn_rows', 'value': int(excel_df['inn_norm'].isna().sum())}
])

display(stats_excel)
display(excel_df[excel_df['inn_norm'].isna()].head(sample_size))


## 2) ССП-справочник из `sandbox_ai.kedr2hive__dbo__active_clients_ul_new`


In [ ]:
sql_ssp = f"""
select
    cast({ssp_inn_col} as string) as inn_raw,
    cast({ssp_zo_col} as string) as zo_raw
from {ssp_table}
"""

with imp:
    ssp_raw = imp.fetch(sql_ssp)

ssp_raw['inn_norm'] = ssp_raw['inn_raw'].apply(normalize_inn)
ssp_raw['zo_norm'] = ssp_raw['zo_raw'].apply(lambda x: None if pd.isna(x) or str(x).strip() == '' else str(x).strip())

ssp_map = ssp_raw[ssp_raw['inn_norm'].notna()].copy()

ssp_card = (
    ssp_map.groupby('inn_norm', as_index=False)
    .agg(
        rows_cnt=('inn_norm', 'count'),
        zo_candidates=('zo_norm', lambda s: int(s.dropna().nunique())),
        zo_non_null_rows=('zo_norm', lambda s: int(s.notna().sum()))
    )
)

ssp_card['zo_status'] = ssp_card['zo_candidates'].apply(
    lambda x: 'zo_missing' if x == 0 else ('zo_unique' if x == 1 else 'zo_conflict')
)

display(ssp_card['zo_status'].value_counts(dropna=False).rename_axis('zo_status').reset_index(name='inn_cnt'))
display(ssp_card[ssp_card['zo_status'] != 'zo_unique'].head(sample_size))


## 3) Покрытие ССП для всех ИНН из Excel


In [ ]:
excel_ssp_cov = excel_inn_unique.merge(ssp_card, on='inn_norm', how='left')

excel_ssp_cov['zo_status'] = excel_ssp_cov['zo_status'].fillna('not_found_in_ssp_table')
excel_ssp_cov['zo_candidates'] = excel_ssp_cov['zo_candidates'].fillna(0).astype(int)
excel_ssp_cov['rows_cnt'] = excel_ssp_cov['rows_cnt'].fillna(0).astype(int)
excel_ssp_cov['zo_non_null_rows'] = excel_ssp_cov['zo_non_null_rows'].fillna(0).astype(int)

coverage = (
    excel_ssp_cov.groupby('zo_status', as_index=False)
    .agg(inn_cnt=('inn_norm', 'count'))
    .sort_values('inn_cnt', ascending=False)
)
coverage['share'] = coverage['inn_cnt'] / coverage['inn_cnt'].sum()

summary = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_unique)},
    {'metric': 'inn_with_zo_unique', 'value': int((excel_ssp_cov['zo_status'] == 'zo_unique').sum())},
    {'metric': 'inn_with_zo_missing', 'value': int((excel_ssp_cov['zo_status'] == 'zo_missing').sum())},
    {'metric': 'inn_with_zo_conflict', 'value': int((excel_ssp_cov['zo_status'] == 'zo_conflict').sum())},
    {'metric': 'inn_not_found_in_ssp_table', 'value': int((excel_ssp_cov['zo_status'] == 'not_found_in_ssp_table').sum())},
])
summary['share'] = summary['value'] / max(len(excel_inn_unique), 1)

display(summary)
display(coverage)


## 4) Детализация проблемных ИНН


In [ ]:
not_found = excel_ssp_cov[excel_ssp_cov['zo_status'] == 'not_found_in_ssp_table'][['inn_norm']].copy()
zo_missing = excel_ssp_cov[excel_ssp_cov['zo_status'] == 'zo_missing'][['inn_norm', 'rows_cnt', 'zo_non_null_rows']].copy()
zo_conflict = excel_ssp_cov[excel_ssp_cov['zo_status'] == 'zo_conflict'][['inn_norm', 'zo_candidates', 'rows_cnt']].copy()

display(not_found.head(sample_size))
display(zo_missing.head(sample_size))
display(zo_conflict.head(sample_size))


In [ ]:
# Расшифровка конфликтов: какие именно zo у конфликтных ИНН
conflict_inn = set(zo_conflict['inn_norm'].tolist())
zo_conflict_detail = ssp_map[ssp_map['inn_norm'].isin(conflict_inn)][['inn_norm', 'zo_norm']].drop_duplicates()
zo_conflict_detail = zo_conflict_detail.sort_values(['inn_norm', 'zo_norm'])

display(zo_conflict_detail.head(sample_size))
print('conflict_detail_rows =', len(zo_conflict_detail))


## 5) Вердикт по применимости источника ССП

Логика интерпретации:
- если `inn_with_zo_unique` покрывает почти все ИНН Excel, источник пригоден как основной;
- `zo_missing` и `not_found_in_ssp_table` — кандидаты на доп. источник/обогащение;
- `zo_conflict` требует правила приоритета (например, latest snapshot или reference-master).
